# 7. Model Inference

In [60]:
import matplotlib.pyplot as plt
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAXResults
import statsmodels.api as sm
from pathlib import Path
import yfinance as yf


In [ ]:
def fetch_gold_yfinance_for_inference(symbol="GC=F", start="2025-01-01"):
    data = yf.download(
        tickers=symbol,
        start=start,
        interval="1d",
        auto_adjust=False,
        progress=True,
        multi_level_index=False
    )
    data = data.reset_index()

    data = data.rename(columns={
        "Date": "date",
        "Close": "close"
    })

    df = data[["date", "close"]].copy()

    df["date"] = pd.to_datetime(df["date"])
    df = df.dropna().sort_values("date").reset_index(drop=True)

    return df


In [62]:
def get_usd_idr_rate():
    fx_data = yf.download(
        tickers="IDR=X",
        period="5d",
        interval="1d",
        auto_adjust=False,
        progress=False,
        multi_level_index=False
    )

    fx_data = fx_data.dropna()

    usd_idr_rate = fx_data["Close"].iloc[-1]

    return float(usd_idr_rate)


In [ ]:
df = fetch_gold_yfinance_for_inference()
usd_idr_rate = get_usd_idr_rate()

latest_actual_troy_usd = df["close"].iloc[-1]
latest_actual_date = df["date"].iloc[-1]

print("Latest actual date:", latest_actual_date)
print("Latest actual gold price, 1 troy ounce USD:", latest_actual_troy_usd)
print("USD/IDR rate:", usd_idr_rate)


[*********************100%***********************]  1 of 1 completed


Latest actual date: 2026-05-06 00:00:00
Latest actual gold price, 1 troy ounce USD: 4696.5
USD/IDR rate: 17400.0


In [ ]:
FORECAST_STEPS = 16
TROY_OUNCE_TO_GRAM = 31.1034768 
sarima_result = SARIMAXResults.load("sarima_final_model.pkl")

In [ ]:
# Forecast future prices using the saved pkl model.
forecast_object = sarima_result.get_forecast(steps=FORECAST_STEPS)
forecast_troy_usd = forecast_object.predicted_mean

model_last_date = sarima_result.model.data.row_labels[-1]
model_last_date = pd.Timestamp(model_last_date).tz_localize(None)

future_dates = pd.bdate_range(
    start=model_last_date + pd.offsets.BDay(1),
    periods=FORECAST_STEPS
)


forecast_df = pd.DataFrame({
    "date": future_dates,
    "predicted_1_troy_usd": forecast_troy_usd.values,
    "predicted_1_gram_usd": forecast_troy_usd.values / TROY_OUNCE_TO_GRAM,
    "predicted_1_troy_idr": forecast_troy_usd.values * usd_idr_rate,
    "predicted_1_gram_idr": (forecast_troy_usd.values / TROY_OUNCE_TO_GRAM) * usd_idr_rate
})

forecast_object_next_day = sarima_result.get_forecast(steps=1)
forecast_next_day_troy_usd = forecast_object_next_day.predicted_mean.iloc[0]
print("\nForecast for the next day:")
print(f"Predicted gold price, 1 troy ounce USD: ${forecast_next_day_troy_usd:.2f}")
print(f"Predicted gold price, 1 gram USD: ${forecast_next_day_troy_usd / TROY_OUNCE_TO_GRAM:.2f}")
print(f"Predicted gold price, 1 troy ounce IDR: Rp{forecast_next_day_troy_usd * usd_idr_rate:.2f}")
print(f"Predicted gold price, 1 gram IDR: Rp{(forecast_next_day_troy_usd / TROY_OUNCE_TO_GRAM) * usd_idr_rate:.2f}")
print("\n Full 7-day forecast:")
forecast_df




Forecast for the next day:
Predicted gold price, 1 troy ounce USD: $4523.43
Predicted gold price, 1 gram USD: $145.43
Predicted gold price, 1 troy ounce IDR: Rp78707701.51
Predicted gold price, 1 gram IDR: Rp2530511.36

 Full 7-day forecast:


e:\Anaconda3\envs\h8_env\lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
e:\Anaconda3\envs\h8_env\lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
e:\Anaconda3\envs\h8_env\lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(


,date,predicted_1_troy_usd,predicted_1_gram_usd,predicted_1_troy_idr,predicted_1_gram_idr
0,2026-05-05,4523.431121,145.431688,7.870770e+07,2.530511e+06
1,2026-05-06,4530.783861,145.668084,7.883564e+07,2.534625e+06
2,2026-05-07,4544.120423,146.096864,7.906770e+07,2.542085e+06
3,2026-05-08,4541.817368,146.022819,7.902762e+07,2.540797e+06
4,2026-05-11,4543.951132,146.091421,7.906475e+07,2.541991e+06
5,2026-05-12,4545.045154,146.126595,7.908379e+07,2.542603e+06
6,2026-05-13,4545.214025,146.132024,7.908672e+07,2.542697e+06
7,2026-05-14,4558.550587,146.560805,7.931878e+07,2.550158e+06
8,2026-05-15,4556.247531,146.486760,7.927871e+07,2.548870e+06
9,2026-05-18,4558.381296,146.555362,7.931583e+07,2.550063e+06


## Inference Conclusion

## Inference Conclusion

- Model final yang digunakan untuk inference adalah SARIMA yang sudah disimpan dalam file `sarima_final_model.pkl`.
- Data terbaru dari yfinance menunjukkan harga aktual emas pada 2026-05-06 sebesar 4,696.50 USD per troy ounce, dengan kurs USD/IDR sebesar 17,400.
- Hasil forecast pertama dari model menghasilkan prediksi harga emas sebesar 4,523.43 USD per troy ounce atau sekitar 145.43 USD per gram.
- Kalau di convert ke rupiah, prediksi tersebut setara dengan sekitar Rp78.71 juta per troy ounce atau Rp2.53 juta per gram.
- Forecast untuk beberapa hari ke depan menunjukkan pergerakan yang cenderung naik secara perlahan, dari sekitar 4,523 USD hingga 4,573 USD per troy ounce.
- Karena model yang digunakan berasal dari data training terakhir yaitu 4 Mei 2026 maka keliatan prediksi sedikit lebih meleset, maka dari itu hasil inference akan lebih optimal jika model rutin di-refit menggunakan data terbaru sebelum digunakan untuk prediksi berikutnya.
